In [1]:
import cv2

cap = cv2.VideoCapture("sample.mp4")

# MOG2という背景差分アルゴリズムを初期化（detectShadows=Trueで影を識別）
fgbg = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=16, detectShadows=True)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    # 背景差分を適用（これだけでマスク画像が作れる）
    fgmask = fgbg.apply(frame)

    # 影（グレー=127）を除外して、動体（白=255）だけを残す
    _, thresh = cv2.threshold(fgmask, 200, 255, cv2.THRESH_BINARY)

    # 輪郭を見つける
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    for contour in contours:
        if cv2.contourArea(contour) < 500: # 感度調整
            continue
        x, y, w, h = cv2.boundingRect(contour)
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

    cv2.imshow('MOG2 Motion Detection', frame)
    if cv2.waitKey(40) == ord('q'): break

cap.release()
cv2.destroyAllWindows()